# ⚔️ Challenge Section

ML_Workflow_DVC.ipynb is our group's main notebook.

This is the version implemented by Ce Chen.

## 🔥 Challenge 1: Activation Function

In this challenge, I changed the activation function in `params.yaml`, and I also updated `train.py` so the model can read different activation names from the YAML file.  
This way, I do not need to hard-code ReLU, Sigmoid, or GELU inside the model.

### `params.yaml`

### `train.py`

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset


def load_params():
    with open("params.yaml", "r") as f:
        return yaml.safe_load(f)


def get_activation(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    elif name == "sigmoid":
        return nn.Sigmoid()
    elif name == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unsupported activation: {name}")


class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, activation_name):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            get_activation(activation_name),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


def get_optimizer(model, params):
    opt_name = params["training"]["optimizer"].lower()
    lr = params["training"]["lr"]
    momentum = params["training"].get("momentum", 0.0)

    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr)
    elif opt_name == "momentum":
        return optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    elif opt_name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {opt_name}")


def load_data():
    train_data = torch.load("data/processed/train.pt")
    test_data = torch.load("data/processed/test.pt")

    X_train, y_train = train_data
    X_test, y_test = test_data

    X_train = X_train.view(X_train.size(0), -1).float() / 255.0
    X_test = X_test.view(X_test.size(0), -1).float() / 255.0

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    return train_dataset, test_dataset


def train():
    params = load_params()

    input_size = params["model"]["input_size"]
    hidden_size = params["model"]["hidden_size"]
    num_classes = params["model"]["num_classes"]
    activation = params["model"]["activation"]

    batch_size = params["training"]["batch_size"]
    epochs = params["training"]["epochs"]

    train_dataset, test_dataset = load_data()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleNet(input_size, hidden_size, num_classes, activation)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model, params)

    gradient_means = []
    gradient_stds = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()

            batch_grad_values = []
            for name, param in model.named_parameters():
                if param.grad is not None:
                    grad_mean = param.grad.mean().item()
                    grad_std = param.grad.std().item()
                    batch_grad_values.append((grad_mean, grad_std))

            if batch_grad_values:
                gradient_means.append(sum(x[0] for x in batch_grad_values) / len(batch_grad_values))
                gradient_stds.append(sum(x[1] for x in batch_grad_values) / len(batch_grad_values))

            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.tolist())
            all_labels.extend(batch_y.tolist())

    accuracy = accuracy_score(all_labels, all_preds)

    Path("models").mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), "models/model.pt")

    metrics = {
        "accuracy": float(accuracy)
    }

    with open("metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    grad_stats = {
        "grad_mean": float(sum(gradient_means) / len(gradient_means)) if gradient_means else 0.0,
        "grad_std": float(sum(gradient_stds) / len(gradient_stds)) if gradient_stds else 0.0
    }

    with open("gradient_stats.json", "w") as f:
        json.dump(grad_stats, f, indent=2)

    print("Training complete.")
    print("Accuracy:", accuracy)
    print("Gradient stats saved to gradient_stats.json")


if __name__ == "__main__":
    train()


### Talking Points

- In this part, I changed the activation from the YAML file instead of changing the Python code every time. I think this is more suitable for DVC experiments.
- ReLU usually gave me a stable result and trained faster. It is simple, so it is a good baseline choice.
- Sigmoid worked, but the training felt slower and the final accuracy was a bit lower. I think one reason is that sigmoid can make gradients smaller.
- GELU also worked well, and in some runs the result was slightly better than ReLU. It looks smoother, so the model may learn a little better.
- From this experiment, I learned that activation function is not just a small detail. It can really affect both convergence speed and final performance.

## 🔥 Challenge 2: Optimizer

In this challenge, I changed the optimizer settings in `params.yaml` and used `train.py` to load the optimizer type automatically.  
I tested `sgd`, `momentum`, and `adam` under the same pipeline idea.

### `params.yaml`

### `train.py`

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset


def load_params():
    with open("params.yaml", "r") as f:
        return yaml.safe_load(f)


def get_activation(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    elif name == "sigmoid":
        return nn.Sigmoid()
    elif name == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unsupported activation: {name}")


class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, activation_name):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            get_activation(activation_name),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


def get_optimizer(model, params):
    opt_name = params["training"]["optimizer"].lower()
    lr = params["training"]["lr"]
    momentum = params["training"].get("momentum", 0.0)

    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr)
    elif opt_name == "momentum":
        return optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    elif opt_name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {opt_name}")


def load_data():
    train_data = torch.load("data/processed/train.pt")
    test_data = torch.load("data/processed/test.pt")

    X_train, y_train = train_data
    X_test, y_test = test_data

    X_train = X_train.view(X_train.size(0), -1).float() / 255.0
    X_test = X_test.view(X_test.size(0), -1).float() / 255.0

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    return train_dataset, test_dataset


def train():
    params = load_params()

    input_size = params["model"]["input_size"]
    hidden_size = params["model"]["hidden_size"]
    num_classes = params["model"]["num_classes"]
    activation = params["model"]["activation"]

    batch_size = params["training"]["batch_size"]
    epochs = params["training"]["epochs"]

    train_dataset, test_dataset = load_data()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleNet(input_size, hidden_size, num_classes, activation)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model, params)

    gradient_means = []
    gradient_stds = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()

            batch_grad_values = []
            for name, param in model.named_parameters():
                if param.grad is not None:
                    grad_mean = param.grad.mean().item()
                    grad_std = param.grad.std().item()
                    batch_grad_values.append((grad_mean, grad_std))

            if batch_grad_values:
                gradient_means.append(sum(x[0] for x in batch_grad_values) / len(batch_grad_values))
                gradient_stds.append(sum(x[1] for x in batch_grad_values) / len(batch_grad_values))

            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.tolist())
            all_labels.extend(batch_y.tolist())

    accuracy = accuracy_score(all_labels, all_preds)

    Path("models").mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), "models/model.pt")

    metrics = {
        "accuracy": float(accuracy)
    }

    with open("metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    grad_stats = {
        "grad_mean": float(sum(gradient_means) / len(gradient_means)) if gradient_means else 0.0,
        "grad_std": float(sum(gradient_stds) / len(gradient_stds)) if gradient_stds else 0.0
    }

    with open("gradient_stats.json", "w") as f:
        json.dump(grad_stats, f, indent=2)

    print("Training complete.")
    print("Accuracy:", accuracy)
    print("Gradient stats saved to gradient_stats.json")


if __name__ == "__main__":
    train()


### Talking Points

- I think Adam was the easiest optimizer to use because it already worked quite well with the default setting.
- SGD could still train the model, but it was slower to improve. If the number of epochs is small, the result may not look very strong.
- Adding momentum made SGD better than plain SGD. It helped the model move in a more useful direction during training.
- This experiment showed me that optimizer choice has a direct effect on training speed and final accuracy.
- For this assignment, Adam looked like the safest choice, but it was still useful to compare it with the other methods.

## 🔥 Challenge 3: Forward and Backward Pass Analysis

In this challenge, I still used the same `train.py`, but now I focused on the gradient part.  
The code records gradient mean and gradient standard deviation during training, then saves them into `gradient_stats.json`.  
This helped me understand what happened during backward propagation.

### `train.py`

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset


def load_params():
    with open("params.yaml", "r") as f:
        return yaml.safe_load(f)


def get_activation(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    elif name == "sigmoid":
        return nn.Sigmoid()
    elif name == "gelu":
        return nn.GELU()
    else:
        raise ValueError(f"Unsupported activation: {name}")


class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, activation_name):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            get_activation(activation_name),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


def get_optimizer(model, params):
    opt_name = params["training"]["optimizer"].lower()
    lr = params["training"]["lr"]
    momentum = params["training"].get("momentum", 0.0)

    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr)
    elif opt_name == "momentum":
        return optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    elif opt_name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {opt_name}")


def load_data():
    train_data = torch.load("data/processed/train.pt")
    test_data = torch.load("data/processed/test.pt")

    X_train, y_train = train_data
    X_test, y_test = test_data

    X_train = X_train.view(X_train.size(0), -1).float() / 255.0
    X_test = X_test.view(X_test.size(0), -1).float() / 255.0

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    return train_dataset, test_dataset


def train():
    params = load_params()

    input_size = params["model"]["input_size"]
    hidden_size = params["model"]["hidden_size"]
    num_classes = params["model"]["num_classes"]
    activation = params["model"]["activation"]

    batch_size = params["training"]["batch_size"]
    epochs = params["training"]["epochs"]

    train_dataset, test_dataset = load_data()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleNet(input_size, hidden_size, num_classes, activation)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model, params)

    gradient_means = []
    gradient_stds = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()

            batch_grad_values = []
            for name, param in model.named_parameters():
                if param.grad is not None:
                    grad_mean = param.grad.mean().item()
                    grad_std = param.grad.std().item()
                    batch_grad_values.append((grad_mean, grad_std))

            if batch_grad_values:
                gradient_means.append(sum(x[0] for x in batch_grad_values) / len(batch_grad_values))
                gradient_stds.append(sum(x[1] for x in batch_grad_values) / len(batch_grad_values))

            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.tolist())
            all_labels.extend(batch_y.tolist())

    accuracy = accuracy_score(all_labels, all_preds)

    Path("models").mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), "models/model.pt")

    metrics = {
        "accuracy": float(accuracy)
    }

    with open("metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    grad_stats = {
        "grad_mean": float(sum(gradient_means) / len(gradient_means)) if gradient_means else 0.0,
        "grad_std": float(sum(gradient_stds) / len(gradient_stds)) if gradient_stds else 0.0
    }

    with open("gradient_stats.json", "w") as f:
        json.dump(grad_stats, f, indent=2)

    print("Training complete.")
    print("Accuracy:", accuracy)
    print("Gradient stats saved to gradient_stats.json")


if __name__ == "__main__":
    train()


### Talking Points

- During training, the loss kept going down, so it showed that the forward pass and backward pass were working together correctly.
- I checked the gradients and found they were not zero. This means the model parameters were actually being updated.
- Saving the gradient statistics into a JSON file made the process more clear. It was easier to explain the training behavior with real numbers.
- If gradients become too small, learning can become very slow. If gradients become too large, training can be unstable.
- This challenge helped me understand that backpropagation is not only theory. We can actually inspect it in code and use it to analyze the model.